#   Moral Values Data Analysis Project

#   Part 0: Extraction and Loading into the Database

For my purposes, I am going to pretend that I cannot just load the entire time series into memory each time I would like to run some analyses, so instead I am going to pair this analysis project with some relational database usage.
For this purpose, my backend is PostgreSQL and I will be accessing it through Python primarily with the help of the `psycopg` SQL driver for PostgreSQL and `SQLAlchemy` for pythonic database access syntax.

##  Step 0

Copy downloaded raw files into data folder.

In [1]:
#   Load the libraries to do this
# import shutil
import os
import json

#   Source folder locations
windows_filesystem = json.load(open('env.json'))['windows_filesystem']
source_folder = os.path.join(windows_filesystem, 'Downloads')

#   Find and copy WVS-related files
for entry in os.scandir(source_folder):
    if entry.is_file():
        if 'WVS' in entry.name and not entry.name.endswith('.zip'):
            source_path = entry.path
            destination_path = os.path.join('..', 'data', entry.name)
            # shutil.copy(source_path, destination_path)
            del source_path, destination_path
    del entry

#   Clear out all remaining objects
del windows_filesystem, source_folder

##  Step 1: Extract and Load Data

In [2]:
import time
import pandas as pd
import numpy as np
import dfext
import csv
from collections import defaultdict
from typing import Optional, List
from sqlalchemy import create_engine, ForeignKey, SmallInteger, Integer, BigInteger, Double, Float, Numeric, String
from sqlalchemy.orm import sessionmaker, DeclarativeBase, Mapped, mapped_column, relationship

In [3]:
#   Connect to database server
db_url = 'postgresql+psycopg://{username}:{password}@{server}/{database}'.format(**json.load(open('env.json'))['postgresql'])
engine = create_engine(db_url)
Session = sessionmaker(engine)

#   Object-Relational Model base
class Base(DeclarativeBase):
    pass

In [4]:
#   Class for variables list table
class Variables(Base):
    __tablename__ = 'variables'
    variable: Mapped[str] = mapped_column(primary_key=True)
    vargroup: Mapped[Optional[str]]
    label: Mapped[Optional[str]]
    missing: Mapped[Optional[str]]
    values: Mapped[List['ValueLabels']] = relationship(back_populates='value', cascade='all, delete')
    wvs7: Mapped[Optional[str]]
    wvs6: Mapped[Optional[str]]
    wvs5: Mapped[Optional[str]]
    wvs4: Mapped[Optional[str]]
    wvs3: Mapped[Optional[str]]
    wvs2: Mapped[Optional[str]]
    wvs1: Mapped[Optional[str]]

#   Class for value labels table
class ValueLabels(Base):
    __tablename__ = 'value_labels'
    variable: Mapped[str] = mapped_column(ForeignKey('variables.variable', ondelete='CASCADE', onupdate='CASCADE'), primary_key=True)
    value = mapped_column(BigInteger, primary_key=True)
    label: Mapped[Optional[str]]

#   Load unique values for determining data types
data_values = defaultdict(set)
with open('../data/WVS_TimeSeries_4_0.csv', 'rt', encoding='utf-8') as f:
    csvreader = csv.DictReader(f, delimiter=',', quotechar='"')
    for r in csvreader:
        for k, v in r.items():
            data_values[k].add(v)
            del k, v
        del r
    del csvreader
    f.close()
del f

#   Instantiate data table
class DataTable(Base):
    __tablename__ = 'data_table'
    S007 = mapped_column(BigInteger, primary_key=True)

#   Map numpy types to SQL types
np_sql_map = {
    np.int64: BigInteger,
    np.int32: Integer,
    np.int16: Integer,
    np.int8: SmallInteger,
    np.float64: Double,
    np.float32: Float,
    np.float16: Numeric,
    np.object_: String
}

#   Populate data table
for k, v in data_values.items():
    if k in 'S007':
        continue
    v = pd.to_numeric(pd.Series(list(v)), downcast='integer', errors='ignore')
    v_hasna = v.isna().sum() > 0
    v_dtype = pd.to_numeric(v.dropna(), downcast='integer', errors='ignore').dtype
    setattr(DataTable, k, mapped_column(np_sql_map[v_dtype.type], nullable=v_hasna))
    del k, v, v_hasna, v_dtype
del np_sql_map, data_values

#   Recreate all tables
Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)

Load variables list

In [5]:
#   Extract data -- Waves administered
df = pd.read_excel('../data/F00003844-WVS_Time_Series_List_of_Variables_and_equivalences_1981_2022_v3_1.xlsx', sheet_name='Hoja1')
df = df.dropna(how='all', axis=1)
df = df.rename(columns=lambda c: c.strip().lower())

#   Extract data -- Variable group and missing values
cb = pd.read_excel('../data/F00003843_WVS_EVS_Integrated_Dictionary_Codebook_v_2014_09_22.xls', sheet_name='Codebook', skiprows=3)
cb = cb.dropna(how='all', axis=1)
cb = cb.rename(columns=lambda c: c.strip().lower())
##  Categories (value labels) will be saved in a separate table
cb = cb.drop(columns=['type','length','categories'])
##  Merge variable group headers to their variables
cb = cb.join(other=cb['idx'].str.split('_', n=1, expand=True).rename(columns=lambda c: f'idx{c+1}'))
cb['vargroup'] = cb['label'].where(cb['idx2'].isna()).ffill()
cb = cb.loc[cb['variable'].notna(),:]
cb = cb.drop(columns=['idx','idx1','idx2'])
cb = cb.reorder_before(['vargroup'], 'label')
##  Join waves administered and variable groups info
cb = cb.merge(right=df, on=['variable'], how='outer')
##  Merge "title" and "label" columns (repeated information)
cb = cb.reorder_after(['missing'], 'title')
cb['title'] = cb['title'].mask(cb['title']==cb['label'])
for c in ['A', 'C', 'D', 'E', 'F', 'S', 'X']:
    cb['label'] = cb['label'].mask(cb['title'].notna()&cb['variable'].str.startswith(c), cb['title'])
cb = cb.drop(columns=['title'])
cb['label'] = cb['label'].str.replace('Llying', 'Lying', regex=False)
cb['label'] = cb['label'].str.replace('Tthe', 'The', regex=False)
##  Fill variable group info for variables only in the waves administered dataset
cb = cb.sort_values(by=['variable'])
cb['variable1'] = cb['variable'].str.replace(r'\d.*', '', regex=True)
cb['vargroup'] = cb.groupby(by=['variable1'])['vargroup'].ffill()
cb = cb.drop(columns=['variable1'])

#   Load variables data to database
cb.to_sql('variables', con=engine, if_exists='append', index=False)

-1

Load variables codebook

In [6]:
#   Extract data -- 
cb = pd.read_excel('../data/F00003843_WVS_EVS_Integrated_Dictionary_Codebook_v_2014_09_22.xls', sheet_name='Codebook', skiprows=3)
cb = cb.dropna(how='all', axis=1)
cb = cb.rename(columns=lambda c: c.strip().lower())
cb['categories'] = cb['categories'].apply(lambda c: dict(l.strip().split(':', 1) for l in c.strip().split('\n') if ':' in l) if c == c and ':' in c else '')
cb['categories'] = cb['categories'].mask(cb['categories'].str.len().eq(0))
cb = cb.join(other=pd.DataFrame(cb['categories'].to_dict()).transpose())
cb = cb.drop(columns=['idx','label','missing','type','length','categories'])
cb = cb.loc[cb['variable'].notna(),:]
cb = cb.set_index(keys=['variable'], append=False)
cb = cb.rename_axis(columns=['value'])
cb = cb.stack().rename('label')
cb = cb.reset_index(drop=False)
cb = cb.apply(pd.to_numeric, downcast='integer', errors='ignore')
for sheet, varlist in {
    'Education': ['X025CS'],
    'Ethnic group': ['X051'],
    'Income scales': ['X047CS'],
    'Language': ['G016','S016'],
    'Political Parties': [],
    'Party preferences': ['E179','E180','E181','E182','E256'],
    'Region': ['X048'],
    'Size of town': ['X049CS'],
}.items():
    for var in varlist:
        col = pd.read_excel('../data/F00003843_WVS_EVS_Integrated_Dictionary_Codebook_v_2014_09_22.xls', sheet_name=sheet, header=None, names=['value','label'])
        col['variable'] = var
        col = col.reorder_first(['variable'])
        cb = pd.concat([cb, col], axis=0)
        del var, col
    del varlist, sheet
cb = cb.sort_values(by=['variable','value'])

#   Load variables data to database
cb.to_sql('value_labels', con=engine, if_exists='append', index=False)

/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/pandas/core/dtypes/cast.py:375: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)


-1

Load data

In [7]:
with pd.read_csv('../data/WVS_TimeSeries_4_0.csv', iterator=True, chunksize=1_000) as csvfile:
    for csvchunk in csvfile:
        uploaded = False
        while not uploaded:
            time.sleep(1)
            try:
                csvchunk.to_sql('data_table', con=engine, if_exists='append', index=False)
                uploaded = True
            except Exception as e:
                print(e)
    csvfile.close()
    del csvfile, csvchunk, uploaded

End SQL session

In [8]:
Session.close_all()
engine.dispose()

/tmp/ipykernel_1117/4233946287.py:1: SADeprecationWarning: The Session.close_all() method is deprecated and will be removed in a future release.  Please refer to session.close_all_sessions(). (deprecated since: 1.3)
  Session.close_all()
